In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/squad-tqa-gemma-2-2b-it-layer20-activations/SQuAD_TQA_gemma_2_2b_it_layer20_activations.pt


In [2]:
!pip install FrEIA pyvene transformers torch nnsight
!pip install --upgrade wandb

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 7.4 MB/s eta 0:00:00
  Created wheel for FrEIA: filename=FrEIA-0.2-py3-none-any.whl size=42763 sha256=2143da6e4792769a329f85e56b028cd9d49358531bf4f4e573f1da0d262ca84c
  Stored in directory: /root/.cache/pip/wheels/ae/40/63/f30f17a00a5c53f982c5c222995b53fe1ee510d2ea13b00856
Successfully built FrEIA
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 74.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: wandb
    Found existing installation: wandb 0.22.2
    Uninstalling wandb-0.22.2:
      Successfully uninstalled wandb-0.22.2


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from FrEIA.framework import SequenceINN
from FrEIA.modules import AllInOneBlock
import pyvene as pv
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
login(new_session = False)
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wandb_api_key")
wandb.login(key=secret_value_0)

2026-02-04 14:12:18.246943: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770214338.437537      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770214338.492181      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770214338.961193      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770214338.961246      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770214338.961249      55 computation_placer.cc:177] computation placer alr

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: da24b025 (jerrycloud3316-ai-club-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
def subnet_fc(dims_in, dims_out):
    """Sub-network for coupling layers: standard MLP with non-linearities."""
    return nn.Sequential(
        nn.Linear(dims_in, 128), 
        nn.GELU(),
        nn.Linear(128, dims_out)
    )

class InvertibleUncertaintyAE(nn.Module):
    def __init__(self, input_dim=2304, bottleneck_dim=128):
        super().__init__()
        self.input_dim = input_dim
        self.bottleneck_dim = bottleneck_dim
        
        # 1. Define the Invertible Neural Network (INN)
        # We use FrEIA to build a flow that is mathematically invertible by construction.
        self.inn = SequenceINN(input_dim)
        for _ in range(4): # Stack 4 coupling blocks
            self.inn.append(AllInOneBlock, subnet_constructor=subnet_fc, permute_soft=False)
            
        # 2. Semantic Entropy Probe (Binary Classifier)
        # Changed to output logits for Binary Classification
        self.se_probe = nn.Sequential(
            nn.Linear(bottleneck_dim, 1) # Outputs raw logits (no Sigmoid here, use BCEWithLogitsLoss)
        )

    def forward(self, x):
        # --- Encoder Path ---
        # Map input x to latent space z (bijective)
        z, log_det_jac = self.inn(x)
        
        # Split z into the bottleneck (informative) and residuals (noise)
        y = z[:, :self.bottleneck_dim]
        z_res = z[:, self.bottleneck_dim:]
        
        # --- Decoder / Reconstruction Path ---
        # We force the reconstruction to rely ONLY on y by zeroing out z_res.
        # This acts as the "bottleneck" constraint.
        z_bottlenecked = torch.cat([y, torch.zeros_like(z_res)], dim=1)
        
        # Invert the INN to get the reconstruction
        x_recon, _ = self.inn(z_bottlenecked, rev=True)
        
        # --- Classification Path ---
        # Predict binary uncertainty from the informative bottleneck y
        se_logits = self.se_probe(y)
        
        return x_recon, se_logits, z_res, log_det_jac

    def predict_uncertainty(self, x):
        """Helper for inference to get actual probabilities/labels"""
        self.eval()
        with torch.no_grad():
            _, logits, _, _ = self.forward(x)
            probs = torch.sigmoid(logits)
            return probs > 0.5  # Returns Boolean tensor

In [9]:
def compute_isometric_loss(model, x, num_vectors=1):
    """
    Efficiently penalizes non-isometry using the Hutchinson estimator.
    Approximates ||J^T J - I||_F without computing the full Jacobian.
    Cost: Only 'num_vectors' extra backward passes (usually 1 is enough).
    """
    x.requires_grad_(True)
    
    # 1. Forward pass to get output z
    z, _ = model.inn(x)
    
    loss_iso = 0
    
    for _ in range(num_vectors):
        # 2. Sample a random unit vector v of same shape as z
        v = torch.randn_like(z)
        v = v / torch.norm(v, dim=1, keepdim=True)
        
        # 3. Compute vector-Jacobian product (v^T J)
        # We want to measure how much J stretches v.
        # Since standard PyTorch calculates v^T J (gradients), we use that.
        # Ideally, if J is orthogonal, ||v^T J|| should be close to ||v|| = 1.
        
        v_J = torch.autograd.grad(
            outputs=z, 
            inputs=x, 
            grad_outputs=v, 
            create_graph=True, 
            retain_graph=True,
            only_inputs=True
        )[0]
        
        # 4. Isometry condition: The norm of the gradient should be 1
        # If J is orthogonal, it preserves the norm of v.
        norm_v_J = torch.norm(v_J, dim=1)
        loss_iso += torch.mean((norm_v_J - 1.0) ** 2)

    return loss_iso / num_vectors

In [11]:
file_path = "/kaggle/input/squad-tqa-gemma-2-2b-it-layer20-activations/SQuAD_TQA_gemma_2_2b_it_layer20_activations.pt"
print(f"Loading data from {file_path}...")

data_dict = torch.load(file_path)
raw_activations = data_dict["activations"].to(torch.float32) # Ensure Float32
act_mean = raw_activations.mean(dim=0)
act_std = raw_activations.std(dim=0) + 1e-6 # Prevent div by zero

dataset_activations = (raw_activations - act_mean) / act_std
raw_labels = data_dict["labels"].to(torch.float32)
# 3. Smart Label Handling (The Fix)
unique_values = torch.unique(raw_labels)
if len(unique_values) == 2 and 0 in unique_values and 1 in unique_values:
    print("Labels are already binary (0/1). Using them directly.")
    binary_targets = raw_labels
    # If dimensions are wrong [Batch], fix to [Batch, 1]
    if binary_targets.dim() == 1:
        binary_targets = binary_targets.unsqueeze(1)
else:
    print("Labels appear to be raw scores. Binarizing now...")
    se_threshold = raw_labels.median()
    binary_targets = (raw_labels > se_threshold).float().unsqueeze(1)

# Verify we have both classes
print(f"Number of Certain samples (0): {(binary_targets == 0).sum().item()}")
print(f"Number of Uncertain samples (1): {(binary_targets == 1).sum().item()}")

Loading data from /kaggle/input/squad-tqa-gemma-2-2b-it-layer20-activations/SQuAD_TQA_gemma_2_2b_it_layer20_activations.pt...
Labels are already binary (0/1). Using them directly.
Number of Certain samples (0): 96
Number of Uncertain samples (1): 104


In [13]:
# --- 1. CONFIGURATION ---
# Define all hypers in a dictionary for WandB tracking
config = {
    "project_name": "gemma-steering-ae",
    "run_name": "isometric_ae_v2",
    "input_dim": 2304,    # Gemma-2b hidden dim
    "bottleneck_dim": 128,
    "batch_size": 32,
    "lr": 1e-3,
    "epochs": 1000,
    "alpha_recon": 1.0,   # Reconstruction Weight
    "beta_class": 5.0,    # Steering/Classification Weight
    "gamma_iso": 0.1,     # Isometry Weight
    "delta_bottle": 0.01, # Bottleneck Sparsity Weight
    "grad_clip": 1.0
}

# --- 2. INIT WANDB ---
wandb.init(
    project=config["project_name"],
    name=config["run_name"],
    config=config
)

# --- 3. MODEL SETUP ---
# (Assuming InvertibleUncertaintyAE class is already defined from previous steps)
iae = InvertibleUncertaintyAE(
    input_dim=config["input_dim"], 
    bottleneck_dim=config["bottleneck_dim"]
).cuda()

# Monitor gradients and parameters to detect exploding/vanishing grads
wandb.watch(iae, log="all", log_freq=50)

optimizer = torch.optim.Adam(iae.parameters(), lr=config["lr"])

# Load Data (Assuming dataset_activations/binary_targets exist)
dataset = TensorDataset(dataset_activations, binary_targets)
dataloader = DataLoader(dataset, batch_size=config["batch_size"], shuffle=True)

print("Starting WandB Training Run...")

# --- 4. TRAINING LOOP ---
for epoch in range(config["epochs"]):
    total_loss_epoch = 0
    
    for batch_i, (batch_x, batch_y) in enumerate(dataloader):
        optimizer.zero_grad()
        
        batch_x = batch_x.cuda()
        batch_y = batch_y.cuda()
        
        # Forward
        x_recon, se_logits, z_res, _ = iae(batch_x)
        
        # Losses
        l_recon = nn.MSELoss()(x_recon, batch_x)
        l_class = nn.BCEWithLogitsLoss()(se_logits, batch_y)
        l_bottleneck = torch.mean(z_res**2)
        l_iso = compute_isometric_loss(iae, batch_x) # Your Hutchinson estimator
        
        # Weighted Sum
        loss = (config["alpha_recon"] * l_recon) + \
               (config["beta_class"] * l_class) + \
               (config["gamma_iso"] * l_iso) + \
               (config["delta_bottle"] * l_bottleneck)
               
        loss.backward()
        
        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(iae.parameters(), config["grad_clip"])
        
        optimizer.step()
        total_loss_epoch += loss.item()

        # Calculate Accuracy for logging
        with torch.no_grad():
            preds = (torch.sigmoid(se_logits) > 0.5).float()
            acc = (preds == batch_y).float().mean()

        # --- 5. LOGGING STEP ---
        # Log every batch (or every N batches) for high resolution
        wandb.log({
            "epoch": epoch,
            "loss/total": loss.item(),
            "loss/reconstruction": l_recon.item(),
            "loss/classification": l_class.item(),
            "loss/isometry": l_iso.item(),
            "loss/bottleneck": l_bottleneck.item(),
            "metrics/accuracy": acc.item()
        })

    # Log average epoch metrics
    avg_loss = total_loss_epoch / len(dataloader)
    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | Loss (Iso): {l_iso:.4f}")

    # --- 6. SAVE CHECKPOINTS TO WANDB ---
    # Save every 10 epochs or if it's the last one
    if epoch % 100 == 0 or epoch == config["epochs"] - 1:
        checkpoint_path = f"checkpoint_epoch_{epoch}.pt"
        # Upload to WandB cloud
        wandb.save(checkpoint_path)

model_path = '/kaggle/working/iae_model.pt'
torch.save({
    'epochs': 1000,
    'layer': 20,
    'llm_id': 'google/gemma-2-2b-it',
    'model_state_dict': iae.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}, model_path)
print("Training Complete.")
wandb.finish()

epoch,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
loss/bottleneck,▇▆▇█▆▄▄▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/classification,▇██▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/isometry,▁▃█▆▂▂▄▂▃▂▃▁▂▂▁▄▂▂▂▃▂▁▂▂▂▂▁▁▂▂▂▂▁▂▃▂▂▂▂▁
loss/reconstruction,█▇▆▆▅▄▄▃▃▄▂▃▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/total,█▅▅▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,171
loss/bottleneck,0.05639
loss/classification,3e-05
loss/isometry,0.01379


Starting WandB Training Run...
Epoch 0 | Loss: 4.2099 | Acc: 0.5000 | Loss (Iso): 0.0001
Epoch 100 | Loss: 0.2450 | Acc: 1.0000 | Loss (Iso): 0.0064
Epoch 200 | Loss: 0.0905 | Acc: 1.0000 | Loss (Iso): 0.0084
Epoch 300 | Loss: 0.0672 | Acc: 1.0000 | Loss (Iso): 0.0020
Epoch 400 | Loss: 0.0635 | Acc: 1.0000 | Loss (Iso): 0.0036
Epoch 500 | Loss: 0.0640 | Acc: 1.0000 | Loss (Iso): 0.0079
Epoch 600 | Loss: 0.0589 | Acc: 1.0000 | Loss (Iso): 0.0050
Epoch 700 | Loss: 0.0564 | Acc: 1.0000 | Loss (Iso): 0.0023
Epoch 800 | Loss: 0.8275 | Acc: 1.0000 | Loss (Iso): 0.0160
Epoch 900 | Loss: 0.0560 | Acc: 1.0000 | Loss (Iso): 0.0028
Training Complete.


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇█
loss/bottleneck,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/classification,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/isometry,█▄▃▂▂▃▃▃▃▃▃▂▃▂▂▂▁▂▂▂▂▂▂▂▁▁▁▁▁▂▂▂▁▂▂▁▁▁▁▂
loss/reconstruction,█▇▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▂▁▁▁▁▁▁
loss/total,█▅▄▄▃▂▂▂▂▂▂▁▂▂▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,999
loss/bottleneck,0.02202
loss/classification,1e-05
loss/isometry,0.00195


In [14]:
def get_iae_steering_vector(iae_model):
    """
    Extracts the steering vector directly from the linear classifier weights.
    Returns the vector in LATENT space (Dimension = bottleneck_dim).
    """
    # Get the weight tensor
    w = iae_model.se_probe[0].weight.detach().data
    
    # Normalize
    # This ensures 'strength=1.0' always implies the same magnitude of shift
    w_normalized = w / torch.norm(w)
    
    print(f"Extracted Vector. Shape: {w_normalized.shape} (Latent Space)")
    return w_normalized

# --- HOW TO RUN IT ---
# Ensure you pass the BINARY targets you created in the loading step
try:
    steer_vec = get_iae_steering_vector(iae)
    print(f"Vector Norm: {torch.norm(steer_vec)}")
    save_path = "/kaggle/working/iae_steer_vec.pt"

    torch.save({
        "vector": steer_vec.detach().cpu(),  # Move to CPU for portability
        "layer_idx": 20,                        # IMPORTANT: Remember the layer!
        "model_id": "google/gemma-2-2b-it",
    }, save_path)
except ValueError as e:
    print(e)

Extracted Vector. Shape: torch.Size([1, 128]) (Latent Space)
Vector Norm: 0.9999999403953552
